In [0]:
CREATE WIDGET TEXT catalog_name DEFAULT "workspace";

CREATE WIDGET TEXT source_file_name DEFAULT "creditcard.csv";

CREATE WIDGET TEXT source_path
DEFAULT "s3://intern-final-project-fraud-detection/landing/creditcard.csv";

CREATE WIDGET TEXT batch_id DEFAULT "batch_001";

CREATE WIDGET TEXT content_sha256 DEFAULT "";

CREATE WIDGET TEXT source_etag DEFAULT "";

CREATE WIDGET TEXT file_size_bytes DEFAULT "";

In [0]:
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_name || '.bronze');

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_name || '.ops');

CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog_name || '.ops.processed_files') (
    source_file_name STRING,
    source_s3_uri STRING,
    source_etag STRING,
    file_size_bytes BIGINT,
    content_sha256 STRING,
    batch_id STRING,
    pipeline_run_id STRING,
    status STRING,
    source_row_count BIGINT,
    bronze_row_count BIGINT,
    started_ts TIMESTAMP,
    completed_ts TIMESTAMP,
    error_message STRING,
    rejection_reason STRING,
    routed_s3_uri STRING,
    target_table STRING
)
USING DELTA;

In [0]:
CREATE OR REPLACE TEMP VIEW bronze_source_raw AS
SELECT *
FROM read_files(
    :source_path,
    format => 'csv',
    header => true,
    inferSchema => true
);

source_row_count
284807


In [0]:
CREATE OR REPLACE TEMP VIEW bronze_batch AS
SELECT
    src.*,

    current_timestamp() AS load_ts,
    :source_file_name AS source_file,
    :source_path AS source_s3_uri,
    :batch_id AS batch_id,
    current_date() AS ingestion_date,
    CONCAT('bronze_', :batch_id) AS pipeline_run_id

FROM bronze_source_raw AS src;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5756789266127700>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE OR REPLACE TEMP VIEW bronze_batch AS\nSELECT\n    src.*,\n\n    current_timestamp() AS load_ts,\n    :source_file_name AS source_file,\n    :source_path AS source_s3_uri,\n    :batch_id AS batch_id,\n    current_date() AS ingestion_date,\n    CONCAT('bronze_', :batch_id) AS pipeline_run_id\n\nFROM bronze_source_raw AS src;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the l

In [0]:
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog_name || '.bronze.transactions_raw')
USING DELTA
AS
SELECT *
FROM bronze_batch
WHERE 1 = 0;

INSERT INTO IDENTIFIER(:catalog_name || '.ops.processed_files') (
    source_file_name,
    source_s3_uri,
    source_etag,
    file_size_bytes,
    content_sha256,
    batch_id,
    pipeline_run_id,
    status,
    source_row_count,
    bronze_row_count,
    started_ts,
    completed_ts,
    error_message,
    rejection_reason,
    routed_s3_uri,
    target_table
)
SELECT
    :source_file_name,
    :source_path,
    NULLIF(:source_etag, ''),
    CASE
        WHEN :file_size_bytes = '' THEN NULL
        ELSE CAST(:file_size_bytes AS BIGINT)
    END,
    NULLIF(:content_sha256, ''),
    :batch_id,
    CONCAT('bronze_', :batch_id),
    'RUNNING',
    (SELECT COUNT(*) FROM bronze_source_raw),
    NULL,
    current_timestamp(),
    NULL,
    NULL,
    NULL,
    NULL,
    :catalog_name || '.bronze.transactions_raw'

WHERE NOT EXISTS (
    SELECT 1
    FROM IDENTIFIER(:catalog_name || '.ops.processed_files')
    WHERE source_s3_uri = :source_path
      AND batch_id = :batch_id
      AND status IN ('RUNNING', 'SUCCESS')
);

num_affected_rows,num_inserted_rows
0,0


In [0]:
INSERT INTO IDENTIFIER(:catalog_name || '.bronze.transactions_raw')
SELECT *
FROM bronze_batch
WHERE NOT EXISTS (
    SELECT 1
    FROM IDENTIFIER(:catalog_name || '.bronze.transactions_raw')
    WHERE source_s3_uri = :source_path
      AND batch_id = :batch_id
);

num_affected_rows,num_inserted_rows
0,0


In [0]:
UPDATE IDENTIFIER(:catalog_name || '.ops.processed_files')

SET
    status = 'SUCCESS',

    bronze_row_count = (
        SELECT COUNT(*)
        FROM IDENTIFIER(:catalog_name || '.bronze.transactions_raw')
        WHERE source_s3_uri = :source_path
          AND batch_id = :batch_id
    ),

    completed_ts = current_timestamp(),

    error_message = NULL

WHERE source_s3_uri = :source_path
  AND batch_id = :batch_id
  AND status = 'RUNNING';

num_affected_rows
0
